# Phase 3: Data Preparation
**CRISP-DM Purpose:** Transform raw data into modeling-ready form. All cleaning, feature engineering, and splitting decisions are made and documented here.

---

In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, str(Path('..').resolve()))

from src.config import SEED, TEST_SIZE, AT_RISK_DAYS, TARGET
from src.features import build_donation_features, add_label
from src.modeling import build_preprocessor

from sklearn.model_selection import train_test_split

raw_dir = Path('..') / 'data' / 'raw'
fallback_dir = Path('..').parent

def find_csv(name):
    p = raw_dir / name
    return p if p.exists() else fallback_dir / name

supporters = pd.read_csv(find_csv('../../datasets/supporters.csv'), parse_dates=['created_at', 'first_donation_date'])
donations  = pd.read_csv(find_csv('../../datasets/donations.csv'),  parse_dates=['donation_date'])

print(f'Loaded: {len(supporters)} supporters, {len(donations)} donations')

Loaded: 60 supporters, 420 donations


## 3.1 Feature Engineering

In [18]:
# Build joined feature set and label (logic lives in src/features.py)
df = build_donation_features(supporters, donations)
df = add_label(df)

# Training scope: all 60 donors (active + inactive)
# Inactive donors ARE the ground truth positive class — they've already lapsed.
# The model learns behavioral patterns that distinguish inactive from active donors,
# then scores active donors at inference time.
df_model = df.copy().reset_index(drop=True)

print(f'All donors: {len(df_model)}')
print(f'at_risk distribution:\n{df_model[TARGET].value_counts().rename({0:"Active (not at risk)", 1:"Inactive (at risk)"})}')


All donors: 60
at_risk distribution:
at_risk
Inactive (at risk)      47
Active (not at risk)    13
Name: count, dtype: int64


## 3.2 Column Inclusion / Exclusion Report

In [19]:
# Columns excluded and the reason why
EXCLUDE = {
    'supporter_id':             'Identifier — no predictive signal',
    'display_name':             'PII — free-text name',
    'organization_name':        'PII — sparse, mostly null for individuals',
    'first_name':               'PII',
    'last_name':                'PII',
    'email':                    'PII',
    'phone':                    'PII',
    'status':                   '⚠ LEAKAGE — directly encodes the label (Inactive=1, Active=0)',
    'created_at':               'Encoded as supporter_tenure_days',
    'first_donation_date':      'Encoded inside build_donation_features',
    'last_donation_date':       'Raw date — already encoded as days_since_last_donation',
    'first_donation_date_d':    'Raw date — already encoded as donor_tenure_days',
    'days_since_last_donation': '⚠ LEAKAGE — inactive donors have high recency by definition; would trivialise prediction',
    TARGET:                     'Target column',
}

print('Excluded columns and reasons:')
for col, reason in EXCLUDE.items():
    print(f'  {col:<30} → {reason}')

feature_cols = [c for c in df_model.columns if c not in EXCLUDE]
print(f'\nFeature columns ({len(feature_cols)} total):')
print(feature_cols)


Excluded columns and reasons:
  supporter_id                   → Identifier — no predictive signal
  display_name                   → PII — free-text name
  organization_name              → PII — sparse, mostly null for individuals
  first_name                     → PII
  last_name                      → PII
  email                          → PII
  phone                          → PII
  status                         → ⚠ LEAKAGE — directly encodes the label (Inactive=1, Active=0)
  created_at                     → Encoded as supporter_tenure_days
  first_donation_date            → Encoded inside build_donation_features
  last_donation_date             → Raw date — already encoded as days_since_last_donation
  first_donation_date_d          → Raw date — already encoded as donor_tenure_days
  days_since_last_donation       → ⚠ LEAKAGE — inactive donors have high recency by definition; would trivialise prediction
  at_risk                        → Target column

Feature columns (17 total)

In [20]:
X = df_model[feature_cols]
y = df_model[TARGET]

numeric_cols     = X.select_dtypes(include='number').columns.tolist()
categorical_cols = X.select_dtypes(exclude='number').columns.tolist()

print(f'Numeric features    ({len(numeric_cols)}): {numeric_cols}')
print(f'Categorical features ({len(categorical_cols)}): {categorical_cols}')


Numeric features    (12): ['supporter_tenure_days', 'total_donations', 'total_estimated_value', 'mean_estimated_value', 'has_recurring', 'n_campaigns', 'n_channels', 'monetary_donations', 'time_donations', 'inkind_donations', 'donor_tenure_days', 'avg_days_between_donations']
Categorical features (5): ['supporter_type', 'relationship_type', 'region', 'country', 'acquisition_channel']


## 3.3 Train / Test Split

Test set is **frozen here and touched exactly once** — in Phase 5 final evaluation.  
All cross-validation and hyperparameter tuning in Phase 4 uses `X_train` / `y_train` only.

In [21]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=y,   # preserves class ratio in both splits
)

print(f'Train: {len(X_train)} rows  |  Test: {len(X_test)} rows')
print(f'\nTrain at_risk rate: {y_train.mean():.1%}')
print(f'Test  at_risk rate: {y_test.mean():.1%}')

# Verify stratification held
assert abs(y_train.mean() - y_test.mean()) < 0.05, 'Stratification failed — class rates diverged'
print('\n✓ Stratification verified — class rates match within 5%')

Train: 48 rows  |  Test: 12 rows

Train at_risk rate: 79.2%
Test  at_risk rate: 75.0%

✓ Stratification verified — class rates match within 5%


## 3.4 Preprocessing Pipeline

All imputation and scaling is fitted **inside a `sklearn.Pipeline`** — never on the full dataset before splitting.  
This prevents validation statistics from leaking into training.

| Column type | Imputation | Scaling |
|---|---|---|
| Numeric | Median (robust to skew and outliers) | StandardScaler |
| Categorical | Most frequent | OneHotEncoder (handle_unknown='ignore') |

In [22]:
preprocessor = build_preprocessor(numeric_cols, categorical_cols)

# Dry-run fit on train to inspect output shape
X_train_transformed = preprocessor.fit_transform(X_train)
print(f'Preprocessor output shape: {X_train_transformed.shape}')
print(f'  → {len(numeric_cols)} numeric + {X_train_transformed.shape[1] - len(numeric_cols)} encoded categorical columns')

Preprocessor output shape: (48, 34)
  → 12 numeric + 22 encoded categorical columns


## 3.5 Imbalance Check

In [23]:
counts = y_train.value_counts()
ratio  = counts.max() / counts.min()

print(f'Training class counts:\n{counts.rename({0:"Not at risk", 1:"At risk"})}')
print(f'\nImbalance ratio: {ratio:.1f}:1')

if ratio >= 4:
    print('⚠ Imbalance ≥ 4:1 → will use class_weight="balanced" in Phase 4 models.')
    print('  Will report per-class precision/recall (not just overall accuracy).')
    print('  Will include PR curve alongside ROC curve in Phase 5.')
else:
    print('✓ Imbalance < 4:1 — standard training acceptable, but class_weight="balanced" applied as precaution.')

Training class counts:
at_risk
At risk        38
Not at risk    10
Name: count, dtype: int64

Imbalance ratio: 3.8:1
✓ Imbalance < 4:1 — standard training acceptable, but class_weight="balanced" applied as precaution.


## 3.6 Save Processed Splits

In [24]:
processed_dir = Path('..') / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

X_train.assign(**{TARGET: y_train}).to_csv(processed_dir / 'train.csv', index=False)
X_test.assign(**{TARGET: y_test}).to_csv(processed_dir / 'test.csv', index=False)

# Save the feature column list for use in jobs/
import json
with open(processed_dir / 'feature_cols.json', 'w') as f:
    json.dump({
        'feature_cols': feature_cols,
        'numeric_cols': numeric_cols,
        'categorical_cols': categorical_cols,
    }, f, indent=2)

print(f'Saved to {processed_dir}:')
print(f'  train.csv      ({len(X_train)} rows)')
print(f'  test.csv       ({len(X_test)} rows)')
print(f'  feature_cols.json')

Saved to ..\data\processed:
  train.csv      (48 rows)
  test.csv       (12 rows)
  feature_cols.json


## 3.7 Preparation Summary

| Decision | Choice | Reason |
|---|---|---|
| Training scope | All 60 donors (active + inactive) | Inactive donors are ground truth positives — train on real lapse examples |
| Label | `at_risk` = 1 if `status == 'Inactive'` | True confirmed lapse, not a proxy threshold |
| `status` excluded as feature | Yes | Directly encodes the label — leakage |
| `days_since_last_donation` excluded | Yes | Inactive donors have high recency by definition — leakage |
| PII excluded | Yes | Not predictive; privacy |
| Numeric imputation | Median | Robust to right-skewed donation amounts |
| Categorical imputation | Most frequent | Safe default; few missings |
| Encoding | OneHotEncoder (handle_unknown='ignore') | Handles unseen categories at inference |
| Split | 80/20 stratified | Preserves class ratio; test set frozen |
| Imbalance handling | `class_weight='balanced'` in model | Applied in Phase 4 |
| Inference scope | Active donors only | We score donors still in play; inactive are already known |

---
**Proceed to Phase 4: Modeling →** `04_modeling.ipynb`
